In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
import json

In [ ]:
base_path = "segmentation_data"
train_path = os.path.join(base_path, "train")
val_path   = os.path.join(base_path, "val")
test_path  = os.path.join(base_path, "test")


def load_labels(split_path):
    with open(os.path.join(split_path, "labels.json"), 'r') as f:
        data = json.load(f)
    categories = data.get('categories', [])
    id_to_name = {cat['id']: cat['name'] for cat in categories}
    return data, id_to_name

labels_data, id_to_name = load_labels(train_path)
num_classes = len(id_to_name)
print(f"Classes disponibles ({num_classes}): {id_to_name}")

def load_segmentation_data(split_path, img_size=(128,128), max_samples=None):
    images_dir = os.path.join(split_path, "data")
    masks_dir  = os.path.join(split_path, "sem_seg")
    
    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    if max_samples:
        image_files = image_files[:max_samples]
    
    images, masks = [], []
    for img_file in image_files:
        # Charger image
        img_path = os.path.join(images_dir, img_file)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, img_size)
        images.append(img)
        
        # Charger masque
        mask_path = os.path.join(masks_dir, img_file)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, img_size, interpolation=cv2.INTER_NEAREST)
        masks.append(mask)
    
    return np.array(images), np.array(masks)

img_size = (128,128)
X_train, y_train = load_segmentation_data(train_path, img_size=img_size, max_samples=500)
X_val, y_val     = load_segmentation_data(val_path, img_size=img_size, max_samples=100)
X_test, y_test   = load_segmentation_data(test_path, img_size=img_size, max_samples=100)

print(f"Shapes apres chargement:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test: {y_test.shape}")


X_train = X_train.astype('float32') / 255.0
X_val   = X_val.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

print("Valeurs max dans y_train:", np.max(y_train))

num_classes = int(np.max([np.max(y_train), np.max(y_val), np.max(y_test)]) + 1)
print("Nombre de classes:", num_classes)

y_train_onehot = tf.keras.utils.to_categorical(y_train, num_classes)
y_val_onehot   = tf.keras.utils.to_categorical(y_val, num_classes)
y_test_onehot  = tf.keras.utils.to_categorical(y_test, num_classes)

print(f"Shapes après one-hot encoding:")
print(f"y_train_onehot: {y_train_onehot.shape}")
print(f"y_val_onehot:   {y_val_onehot.shape}")
print(f"y_test_onehot:  {y_test_onehot.shape}")


all_masks = np.concatenate([y_train.flatten(), y_val.flatten(), y_test.flatten()])
unique_values, counts = np.unique(all_masks, return_counts=True)
total_pixels = len(all_masks)
class_weights = {val: total_pixels / (len(unique_values) * count) for val, count in zip(unique_values, counts)}

print("Poids par classe:")
for val, weight in class_weights.items():
    print(f"{id_to_name.get(val, f'Classe {val}'):>15}: {weight:.4f}")


def augment_images_masks(images, masks):
    augmented_images, augmented_masks = [], []
    for img, mask in zip(images, masks):
        # flip horizontal
        if np.random.rand() < 0.5:
            img = np.fliplr(img)
            mask = np.fliplr(mask)
        # flip vertical
        if np.random.rand() < 0.5:
            img = np.flipud(img)
            mask = np.flipud(mask)
        # rotation 90°
        if np.random.rand() < 0.5:
            img = np.rot90(img)
            mask = np.rot90(mask)
        augmented_images.append(img)
        augmented_masks.append(mask)
    return np.array(augmented_images), np.array(augmented_masks)


X_train_aug, y_train_aug = augment_images_masks(X_train, y_train_onehot)
print(f"X_train_aug: {X_train_aug.shape}, y_train_aug: {y_train_aug.shape}")


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Metriques
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def iou_metric(y_true, y_pred):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
    return intersection / (union + tf.keras.backend.epsilon())

# Construction du modele
inputs = keras.Input(shape=(128, 128, 3))


# Encoder
c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(c1)
p1 = layers.MaxPooling2D((2, 2))(c1)

c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(p1)
c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(c2)
p2 = layers.MaxPooling2D((2, 2))(c2)

c3 = layers.Conv2D(256, 3, activation='relu', padding='same')(p2)
c3 = layers.Conv2D(256, 3, activation='relu', padding='same')(c3)
p3 = layers.MaxPooling2D((2, 2))(c3)

# Bottleneck
c4 = layers.Conv2D(512, 3, activation='relu', padding='same')(p3)
c4 = layers.Conv2D(512, 3, activation='relu', padding='same')(c4)

# Decoder
u5 = layers.Conv2DTranspose(256, 2, strides=(2, 2), padding='same')(c4)
u5 = layers.concatenate([u5, c3])
c5 = layers.Conv2D(256, 3, activation='relu', padding='same')(u5)
c5 = layers.Conv2D(256, 3, activation='relu', padding='same')(c5)

u6 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(c5)
u6 = layers.concatenate([u6, c2])
c6 = layers.Conv2D(128, 3, activation='relu', padding='same')(u6)
c6 = layers.Conv2D(128, 3, activation='relu', padding='same')(c6)

u7 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(c6)
u7 = layers.concatenate([u7, c1])
c7 = layers.Conv2D(64, 3, activation='relu', padding='same')(u7)
c7 = layers.Conv2D(64, 3, activation='relu', padding='same')(c7)

# Output - dynamique selon num_classes
outputs = layers.Conv2D(num_classes, 1, activation='softmax')(c7)

model = keras.Model(inputs, outputs)



model.summary()


In [ ]:
def weighted_categorical_crossentropy(weights):
    weights = tf.constant(weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_true_f = tf.reshape(y_true, [-1, num_classes])
        y_pred_f = tf.reshape(y_pred, [-1, num_classes])
        loss = -tf.reduce_sum(y_true_f * tf.math.log(y_pred_f + 1e-7) * weights, axis=-1)
        return tf.reduce_mean(loss)
    return loss

weights_array = np.array([class_weights[i] for i in range(num_classes)])
loss_fn = weighted_categorical_crossentropy(weights_array)

model.compile(
    optimizer='adam',
    loss=loss_fn,
    metrics=['accuracy', dice_coefficient, iou_metric]
)


callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
    keras.callbacks.ModelCheckpoint(
        'best_zerowaste_5classes.h5',
        save_best_only=True,
        monitor='val_loss'
    )
]

history = model.fit(
    X_train, y_train_onehot,
    batch_size=8, 
    epochs=8,
    validation_data=(X_val, y_val_onehot),
    callbacks=callbacks,
    verbose=1
)

In [ ]:

best_model = keras.models.load_model('best_zerowaste_5classes.h5', 
                                   custom_objects={
                                       'dice_coefficient': dice_coefficient,
                                       'iou_metric': iou_metric
                                   })


test_results = best_model.evaluate(X_test, y_test_onehot, verbose=0)
print("Résultats globaux sur le test set:")
print(f"Loss: {test_results[0]:.4f}")
print(f"Accuracy: {test_results[1]:.4f}")
print(f"Dice Coefficient: {test_results[2]:.4f}")
print(f"IoU: {test_results[3]:.4f}")

y_pred = best_model.predict(X_test)
y_pred_argmax = np.argmax(y_pred, axis=-1)

In [ ]:
# Visualisation 
fig, axes = plt.subplots(4, 4, figsize=(18, 16))

for i in range(4):
    # Image originale
    axes[i, 0].imshow(X_test[i])
    axes[i, 0].set_title('Image originale')
    axes[i, 0].axis('off')
    
    # Masque vrai
    axes[i, 1].imshow(y_test[i], cmap=cmap_custom, vmin=0, vmax=4)
    axes[i, 1].set_title('Masque vrai')
    axes[i, 1].axis('off')
    
    # Masque predit
    axes[i, 2].imshow(y_pred_argmax[i], cmap=cmap_custom, vmin=0, vmax=4)
    axes[i, 2].set_title('Masque prédit')
    axes[i, 2].axis('off')
    
    # Diff
    diff = np.abs(y_test[i] - y_pred_argmax[i])
    axes[i, 3].imshow(diff, cmap='hot')
    axes[i, 3].set_title('Différence (erreurs)')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()


print("\n" + "="*50)
print("LeGENDE DES CLASSES:")

for class_id, class_name in class_mapping.items():
    print(f"ID {class_id}: {class_name}")



from sklearn.metrics import confusion_matrix, classification_report

y_true_flat = y_test.flatten()
y_pred_flat = y_pred_argmax.flatten()

# Matrice de confusion
cm = confusion_matrix(y_true_flat, y_pred_flat, labels=range(num_classes))

print("Matrice de confusion:")
print("Rows: Vrai, Columns: Predit")
print(" " * 10, end="")
for i in range(num_classes):
    print(f"{i:>5}", end="")
print()
for i in range(num_classes):
    print(f"Classe {i}:", end="")
    for j in range(num_classes):
        print(f"{cm[i,j]:>5}", end="")
    print()

# Rapport de classification
print("\nRapport detaille par classe:")
for class_id in range(num_classes):
    if class_id in class_mapping:
        class_name = class_mapping[class_id]
        true_positives = cm[class_id, class_id]
        false_positives = np.sum(cm[:, class_id]) - true_positives
        false_negatives = np.sum(cm[class_id, :]) - true_positives
        true_negatives = np.sum(cm) - (true_positives + false_positives + false_negatives)
        
        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        print(f"\n{class_name:>15}:")
        print(f"  Precision: {precision:.3f}")
        print(f"  Recall:    {recall:.3f}")
        print(f"  F1-Score:  {f1:.3f}")